# Percentile Interval Extraction
## Input Data
- **Daily NetCDF files** from WRF simulations (15-min temporal resolution), AFWA_TOTPRECIP
- **Historical percentiles dataset** (wet periods > 0.254 mm trace threshold)
- Spatial: Full WRF domain
- Temporal: Water years (Oct 1 to Sep 30)
- Scenarios: HIST, MID4p5, MID8p5, EOC4p5, EOC8p5

## Processing Steps
1. **Define precipitation bands**:
   - **0–0.254 mm** (sub-trace, also QC check)
   - **>0.254 mm**: Percentile-based bands (e.g., trace-10th, 10th-20th, ..., 80th-90th, 90th–99th)
   - Percentiles calculated only from wet periods above 0.254mm
   
2. **Filter by season** (or annual, defined by months):
   - All months or seasonal subset
   - Applied lazily using Dask

3. **Calculate for each band**:
   - **Occurrences**: Count of 15-min periods within band
   - **Accumulations**: Total precipitation (mm) within band
   
4. **Aggregate spatially**:
   - Per grid cell across water year
   - Sub-trace band (0mm-0.254mm) provides consistency check on precipitation frequency

## Output
- **Two NetCDF file types** per band/year/season:
  - `DIST_*`: Occurrence counts (`PRECIP_PERIOD`, 15-minute period count)
  - `Accum_*`: Precipitation sums (`PRECIP_SUM`, mm)
- Used to quantify contribution of different intensity ranges to total precipitation

In [ ]:
#Import Libraries
from __future__ import annotations
# OS packages
import os
import itertools
import glob
import time
import gc
# Data Packages
import numpy as np
import dask.array as da
import xarray as xr
# Plotting Packages
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.offsetbox import AnchoredText
from matplotlib import gridspec, colors, cm, ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cartopy.io.img_tiles as cimgt
from datetime import datetime, timedelta
# MPL inline magic
%matplotlib inline

In [2]:
# Set up Dask Environment
from distributed import Client
# Depending on your workstation specifications, you may need to adjust these values.
# On a single machine, n_workers=1 is usually better.
client = Client(n_workers=4, threads_per_worker=2, memory_limit="80GB") # mem limit * n_workers = total memory allocated
client

2025-05-24 20:04:58,704 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-_a5hu3t8', purging
2025-05-24 20:04:58,705 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-t83ndgl7', purging
2025-05-24 20:04:58,706 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-kqhuvi8h', purging
2025-05-24 20:04:58,711 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-9kp80k6z', purging
2025-05-24 20:04:58,712 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-iemolivm', purging


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 298.02 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42524,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 8
Started: Just now,Total memory: 298.02 GiB
Comm: tcp://127.0.0.1:38651,Total threads: 2
Dashboard: http://127.0.0.1:33791/status,Memory: 74.51 GiB
Nanny: tcp://127.0.0.1:39797,


In [3]:
#  Helper Functions
def filter_valid_file_paths(file_paths):
    """Filter out file paths that don't exist."""
    valid_file_paths = []
    for path in file_paths:
        if os.path.exists(path):
            valid_file_paths.append(path)
    return valid_file_paths

def get_files_in_date_range(path, date_range, hours_offset):
    """Get files within a specified date range."""
    date_format = "%Y-%m-%d"  # Format of the date range string
    
    # Parse the date range string to obtain start and end dates
    start_date_str, end_date_str = date_range.split(" to ")
    start_date = datetime.strptime(start_date_str, date_format)
    end_date = datetime.strptime(end_date_str, date_format)
    
    # Apply the offset to the start and end dates
    hours_offset += 24  # we need to account for the offset coming at the end of the day
    offset_timedelta = timedelta(hours=hours_offset)
    end_date += offset_timedelta
    
    # Initialize the list of files
    files = []
    
    # Iterate over each date in the date range
    current_date = start_date
    while current_date <= end_date:
        folder_year = current_date.year
        year = current_date.year
        month = current_date.month
        day = current_date.day
        
        # Adjust year if the date is after September 30th
        if month < 9 or (month == 9 and day <= 30):
            folder_year -= 1
        
        # Path patterns for different scenarios - the actual path will be selected based on scenario
        if scenario == "HIST":
            fpath = f"{path}/{folder_year}-{folder_year+1}/AFWA_TOTPRECIP_{year}-{month:02d}-{day:02d}.nc"
        elif scenario in ["MID4p5", "MID8p5"]:
            fpath = f"{path}/AFWA_TOTPRECIP_{year}-{month:02d}-{day:02d}.nc"
        elif scenario in ["EOC4p5", "EOC8p5"]:
            fpath = f"{path}/{folder_year}-{folder_year+1}/AFWA_TOTPRECIP_{year}-{month:02d}-{day:02d}.nc"
        
        # Add file path to list
        files.extend([fpath])
        
        # Increment the current date by 1 day
        current_date += timedelta(days=1)
        
    return filter_valid_file_paths(files)

In [4]:
# Main Processing Function
def percentiles_from_daily(run_season, yr_, lower_bound, upper_bound, is_lower_fixed=False, is_upper_fixed=False):
    """
    Process precipitation data for a given percentile or fixed mm band.
    
    Parameters:
    -----------
    run_season : list
        List of months to include (1-12)
    yr_ : int
        Starting year for the water year (Oct 1 to Sep 30)
    lower_bound : float
        Lower bound of the band (percentile between 0-1 or mm threshold)
    upper_bound : float
        Upper bound of the band (percentile between 0-1 or mm threshold)
    is_lower_fixed : bool
        If True, lower_bound is a fixed threshold in mm; if False, it's a percentile (0-1)
    is_upper_fixed : bool
        If True, upper_bound is a fixed threshold in mm; if False, it's a percentile (0-1)
    """
    # Get the start time for performance tracking
    st = time.time()
    
    # Define the date range for the water year
    hours_offset = -1  # Adjust to avoid rolling into the next day
    date_start = f'{yr_}-10-01'
    date_end = f'{yr_+1}-09-30'
    year_range = f'{date_start[:4]}-{date_end[:4]}'
    print(f"Processing year range: {year_range}")
    
    # Create a string representation of months in the season
    months_dict = {1: "J", 2: "F", 3: "M", 4: "A", 5: "M", 6: "J", 
                   7: "J", 8: "A", 9: "S", 10: "O", 11: "N", 12: "D"}
    months_of_data_str = "".join([months_dict[v] for v in run_season]) if len(run_season) < 12 else "ALL"
    print(f"Months: {months_of_data_str}")
    
    # Format the band description for output filenames
    if is_lower_fixed:
        lower_desc = f"{lower_bound:.3f}mm".replace(".", "p")
    else:
        lower_desc = f"p{int(lower_bound*100)}" if lower_bound < 1 else f"p{int(lower_bound)}"
    
    if is_upper_fixed:
        if upper_bound >= 1000:
            upper_desc = "99thplus"  # Use "99thplus" instead of infinity
        else:
            upper_desc = f"{upper_bound:.3f}mm".replace(".", "p")
    else:
        upper_desc = f"p{int(upper_bound*100)}" if upper_bound < 1 else f"p{int(upper_bound)}"
    
    band_desc = f"{lower_desc}-{upper_desc}"
    print(f"Processing band: {band_desc}")
    
    # Get data path based on the chosen scenario
    if scenario == "HIST":
        data_path = "/home/scratch/WRF_BCC/precipitation/AFWA_TOTPRECIP/historical/"
    elif scenario == "MID4p5":
        data_path = "/home/scratch/WRF_BCC/precipitation/AFWA_TOTPRECIP/mid_century_4p5_FIXED/"
    elif scenario == "MID8p5":
        data_path = "/home/scratch/WRF_BCC/precipitation/AFWA_TOTPRECIP/mid_century_8p5_FIXED/"
    elif scenario == "EOC4p5":
        data_path = "/home/scratch/WRF_BCC/precipitation/AFWA_TOTPRECIP/end_of_century_4p5/"
    elif scenario == "EOC8p5":
        data_path = "/home/scratch/WRF_BCC/precipitation/AFWA_TOTPRECIP/end_of_century_8p5/"
    
    # Load daily precipitation dataset files
    files0 = get_files_in_date_range(data_path, f'{date_start} to {date_end}', hours_offset)
    files0.sort()
    print(f'Number of files in dataset 0: {len(files0)}')
    
    # Only load percentiles dataset if needed for non-fixed thresholds
    percentile_file = None
    if not (is_lower_fixed and is_upper_fixed):
        # Load the percentile dataset (only if we have percentile thresholds)
        percentile_path = os.path.join(f'/home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/wet_period_percentiles_base0p254/historical_{months_of_data_str}_quantiles_gt0p254.nc')
        print(f'Percentiles from: {percentile_path}')
        percentile_file = glob.glob(percentile_path, recursive=True)
        percentile_file.sort()
        print(f'Number of percentile files: {len(percentile_file)}')
    
    # Load the WRF grid
    grid = xr.open_dataset(os.path.join("/home/sleake/WRF-BCC_Geogrid", "geo_em.d01.nc"))
    
    # Load the daily precipitation data using Dask for delayed computation
    print("Loading precipitation data...")
    das0 = []
    for f in files0:
        da0 = xr.open_mfdataset(f, chunks='auto', combine='by_coords', engine='netcdf4')
        das0.append(da0)
    
    ds0 = xr.concat(das0, dim="Time")
    
    # Filter the data by season lazily
    if len(run_season) < 12:  # Only filter if not using all months
        ds0 = ds0.where(ds0.Time.dt.month.isin(run_season))
    
    # Prepare threshold values
    print("Preparing threshold values...")
    
    # Handle the different combinations of fixed vs percentile thresholds
    if is_lower_fixed and is_upper_fixed:
        # Both thresholds are fixed values in mm
        quantile_thresh_0 = lower_bound
        quantile_thresh_1 = upper_bound
        print(f"Using fixed thresholds: {quantile_thresh_0}mm to {quantile_thresh_1}mm")
    elif is_lower_fixed and not is_upper_fixed:
        # Lower is fixed mm, upper is percentile
        quantile_thresh_0 = lower_bound
        # Load percentile dataset
        ds1 = xr.open_mfdataset(percentile_file, chunks='auto', combine='by_coords', engine='netcdf4')
        quantile_thresh_1 = ds1.AFWA_TOTPRECIP.sel(quantile=upper_bound, method='nearest')
        print(f"Using fixed lower: {quantile_thresh_0}mm and percentile upper: {upper_bound} ({quantile_thresh_1.values.mean():.3f}mm)")
    elif not is_lower_fixed and is_upper_fixed:
        # Lower is percentile, upper is fixed mm
        ds1 = xr.open_mfdataset(percentile_file, chunks='auto', combine='by_coords', engine='netcdf4')
        quantile_thresh_0 = ds1.AFWA_TOTPRECIP.sel(quantile=lower_bound, method='nearest')
        quantile_thresh_1 = upper_bound
        print(f"Using percentile lower: {lower_bound} ({quantile_thresh_0.values.mean():.3f}mm) and fixed upper: {quantile_thresh_1}mm")
    else:
        # Both are percentiles
        ds1 = xr.open_mfdataset(percentile_file, chunks='auto', combine='by_coords', engine='netcdf4')
        quantile_thresh_0 = ds1.AFWA_TOTPRECIP.sel(quantile=lower_bound, method='nearest')
        quantile_thresh_1 = ds1.AFWA_TOTPRECIP.sel(quantile=upper_bound, method='nearest')
        print(f"Using percentile thresholds: {lower_bound} ({quantile_thresh_0.values.mean():.3f}mm) to {upper_bound} ({quantile_thresh_1.values.mean():.3f}mm)")
    
    # Create boolean masks for the band
    print("Creating masks for the precipitation band...")
    da_bool_period_0 = ds0.AFWA_TOTPRECIP >= quantile_thresh_0
    da_bool_period_1 = ds0.AFWA_TOTPRECIP < quantile_thresh_1
    da_bool_period = da_bool_period_0 & da_bool_period_1
    
    # Calculate occurrences (number of days in the band)
    print("Calculating occurrences...")
    precip_occur = da_bool_period.sum(dim='Time', skipna=True)
    
    # Create occurrence dataset with metadata
    precip_occur.name = 'PRECIP_PERIOD'
    ds_occur = xr.Dataset({'PRECIP_PERIOD': precip_occur})
    
    # Add metadata
    ds_occur.attrs['lower_bound'] = str(lower_bound)
    ds_occur.attrs['upper_bound'] = str(upper_bound)
    ds_occur.attrs['is_lower_fixed'] = str(is_lower_fixed)
    ds_occur.attrs['is_upper_fixed'] = str(is_upper_fixed)
    ds_occur.attrs['band_description'] = band_desc
    ds_occur.attrs['months'] = str(run_season)
    ds_occur.attrs['precip_thresh_for_quantile_calc_mm'] = 0.254
    
    # Calculate accumulations (sum of precipitation in the band)
    print("Calculating accumulations...")
    precip_masked = ds0.AFWA_TOTPRECIP.where(da_bool_period)
    precip_sum = precip_masked.sum(dim='Time', skipna=True)
    
    # Create accumulation dataset with metadata
    precip_sum.name = 'PRECIP_SUM'
    ds_accum = xr.Dataset({'PRECIP_SUM': precip_sum})
    
    # Add metadata
    ds_accum.attrs['lower_bound'] = str(lower_bound)
    ds_accum.attrs['upper_bound'] = str(upper_bound)
    ds_accum.attrs['is_lower_fixed'] = str(is_lower_fixed)
    ds_accum.attrs['is_upper_fixed'] = str(is_upper_fixed)
    ds_accum.attrs['band_description'] = band_desc
    ds_accum.attrs['months'] = str(run_season)
    ds_accum.attrs['precip_thresh_for_quantile_calc_mm'] = 0.254
    
    # Create output directory based on scenario
    output_dir = f"/home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/{scenario}"
    os.makedirs(output_dir, exist_ok=True)
    
    # Save occurrence dataset
    occur_filename = f"DIST_{scenario}_{year_range}_{months_of_data_str}_{band_desc}_0p254base_wet_period.nc"
    occur_path = os.path.join(output_dir, occur_filename)
    print(f"Saving occurrences to: {occur_path}")
    ds_occur.to_netcdf(path=occur_path, mode="w")
    
    # Save accumulation dataset
    accum_filename = f"Accum_{scenario}_{year_range}_{months_of_data_str}_{band_desc}_0p254base_wet_period.nc"
    accum_path = os.path.join(output_dir, accum_filename)
    print(f"Saving accumulations to: {accum_path}")
    ds_accum.to_netcdf(path=accum_path, mode="w")
    
    # Report processing time
    elapsed = time.time() - st
    print(f"Processing time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
    print(f"Current time: {datetime.now().strftime('%H:%M:%S')}")
    print("-" * 50)
    
    # Clean up to release memory
    ds0.close()
    if 'ds1' in locals():
        ds1.close()
    grid.close()
    gc.collect()

In [5]:
# Configuration - Set Scenario and Year Ranges
# CHANGE THESE VARIABLES TO SWITCH SCENARIOS
# Available scenarios: "HIST", "MID4p5", "MID8p5", "EOC4p5", "EOC8p5"
scenario = "EOC4p5"

# Year ranges for each scenario
if scenario == "HIST":
    years = list(range(1990, 2005, 1))
elif scenario in ["MID4p5", "MID8p5"]:
    years = list(range(2040, 2055, 1))
elif scenario in ["EOC4p5", "EOC8p5"]:
    years = list(range(2085, 2100, 1))

print(f"Selected scenario: {scenario}")
print(f"Year range: {years[0]}-{years[-1]+1}")

# Seasons to process
# run_seasons = [[[3,4,5],[6,7,8],[9,10,11],[12,1,2], [1,2,3,4,5,6,7,8,9,10,11,12]]]
run_seasons = [[[3,4,5],
                [6,7,8],
                [9,10,11],
                [12,1,2]
               ]]
# run_seasons = [[[1,2,3,4,5,6,7,8,9,10,11,12]]]  # All months for annual
hours_offset = -1

Selected scenario: EOC4p5
Year range: 2085-2100


In [6]:
# Define Percentile Bands
# Define the percentile bands to process
# Format: (lower_bound, upper_bound, is_lower_fixed, is_upper_fixed)
percentile_bands = [
    # 0mm to <0.254mm
    (0.0, 0.254, True, True),
    
    # 0.254mm to <10th percentile
    (0.254, 0.10, True, False),
    
    # 10th percentile bands
    (0.10, 0.20, False, False),
    (0.20, 0.30, False, False),
    (0.30, 0.40, False, False),
    (0.40, 0.50, False, False),
    (0.50, 0.60, False, False),
    (0.60, 0.70, False, False),
    (0.70, 0.80, False, False),
    (0.80, 0.90, False, False),
    
    # 90th to 99th band
    (0.90, 0.99, False, False),
    
    # ≥99th percentile (using a very large value as upper bound)
    (0.99, 10000, False, True)  
]

# Display configuration
print("Processing plan:")
print(f"Scenario: {scenario}")
print(f"Years: {years[0]}-{years[-1]}")

# Format and display the percentile bands
print("\nPercentile bands to process:")
for i, band in enumerate(percentile_bands):
    lower, upper, is_lower_fixed, is_upper_fixed = band
    
    # Format the band description
    if is_lower_fixed:
        lower_desc = f"{lower:.3f}mm".replace(".", "p")
    else:
        lower_desc = f"p{int(lower*100)}"
        
    if is_upper_fixed:
        if upper >= 1000:
            upper_desc = "99thplus" 
        else:
            upper_desc = f"{upper:.3f}mm".replace(".", "p")
    else:
        upper_desc = f"p{int(upper*100)}"
        
    print(f"{i+1}. {lower_desc} to {upper_desc}")

# Print season information
print("\nSeasons to process:")
for i, season_group in enumerate(run_seasons):
    for j, season in enumerate(season_group):
        season_name = "ALL" if len(season) == 12 else "".join(["JFMAMJJASOND"[m-1] for m in season])
        if len(season) < 12:
            season_desc = ""
            if set(season) == {3, 4, 5}:
                season_desc = " (Spring)"
            elif set(season) == {6, 7, 8}:
                season_desc = " (Summer)"
            elif set(season) == {9, 10, 11}:
                season_desc = " (Fall)"
            elif set(season) == {12, 1, 2}:
                season_desc = " (Winter)"
            print(f"  - {season_name}{season_desc}: months {season}")
        else:
            print(f"  - {season_name} (All months)")

Processing plan:
Scenario: EOC4p5
Years: 2085-2099

Percentile bands to process:
1. 0p000mm to 0p254mm
2. 0p254mm to p10
3. p10 to p20
4. p20 to p30
5. p30 to p40
6. p40 to p50
7. p50 to p60
8. p60 to p70
9. p70 to p80
10. p80 to p90
11. p90 to p99
12. p99 to 99thplus

Seasons to process:
  - MAM (Spring): months [3, 4, 5]
  - JJA (Summer): months [6, 7, 8]
  - SON (Fall): months [9, 10, 11]
  - DJF (Winter): months [12, 1, 2]


In [ ]:
# Execute Processing (optionally with test mode)
# Set to True to process just one combination for testing
test_mode = False

if test_mode:
    # Process just one combination for testing
    test_year = years[0]
    test_season = run_seasons[0][0]  # First season in first group
    test_band = percentile_bands[0]  # First band
    
    # Format the test band for display
    lower, upper, is_lower_fixed, is_upper_fixed = test_band
    if is_lower_fixed:
        lower_desc = f"{lower:.3f}mm".replace(".", "p")
    else:
        lower_desc = f"p{int(lower*100)}"
    if is_upper_fixed:
        if upper >= 1000:
            upper_desc = "99thplus"  # Use "99thplus" instead of infinity
        else:
            upper_desc = f"{upper:.3f}mm".replace(".", "p")
    else:
        upper_desc = f"p{int(upper*100)}"
    
    print(f"TEST MODE: Processing one combination only")
    print(f"Year: {test_year}, Season: {test_season}, Band: {lower_desc}-{upper_desc}")
    
    percentiles_from_daily(
        test_season, 
        test_year, 
        test_band[0], 
        test_band[1], 
        test_band[2], 
        test_band[3]
    )
else:
    # Process all combinations
    results = []
    
    for i, (season_group) in enumerate(run_seasons):
        for sn in season_group:
            season_name = "ALL" if len(sn) == 12 else "".join(["JFMAMJJASOND"[m-1] for m in sn])
            print(f"\nProcessing season: {season_name}")
            
            for band in percentile_bands:
                lower, upper, is_lower_fixed, is_upper_fixed = band
                
                # Format band description for display
                if is_lower_fixed:
                    lower_desc = f"{lower:.3f}mm".replace(".", "p")
                else:
                    lower_desc = f"p{int(lower*100)}"
                if is_upper_fixed:
                    if upper >= 1000:
                        upper_desc = "99thplus"  # Use "99thplus" instead of infinity
                    else:
                        upper_desc = f"{upper:.3f}mm".replace(".", "p")
                else:
                    upper_desc = f"p{int(upper*100)}"
                
                band_desc = f"{lower_desc}-{upper_desc}"
                print(f"\nProcessing band: {band_desc}")
                
                for yr in years:
                    print(f"Processing year: {yr}-{yr+1}")
                    try:
                        percentiles_from_daily(sn, yr, lower, upper, is_lower_fixed, is_upper_fixed)
                        results.append({
                            'year': yr,
                            'season': sn,
                            'band': band_desc,
                            'success': True
                        })
                    except Exception as e:
                        print(f"ERROR processing {yr}, {sn}, {band_desc}: {str(e)}")
                        results.append({
                            'year': yr,
                            'season': sn,
                            'band': band_desc,
                            'success': False,
                            'error': str(e)
                        })
    
    # Print a summary of results
    print("\n" + "="*50)
    print("PROCESSING SUMMARY")
    print("="*50)
    success_count = sum(1 for r in results if r['success'])
    failure_count = len(results) - success_count
    print(f"Total combinations processed: {len(results)}")
    print(f"Successful: {success_count}")
    print(f"Failed: {failure_count}")
    
    if failure_count > 0:
        print("\nFailed combinations:")
        for r in results:
            if not r['success']:
                year = r['year']
                season = r['season']
                season_name = "ALL" if len(season) == 12 else "".join(["JFMAMJJASOND"[m-1] for m in season])
                print(f"- Year: {year}-{year+1}, Season: {season_name}, Band: {r['band']}")
                print(f"  Error: {r['error']}")


Processing season: MAM

Processing band: 0p000mm-0p254mm
Processing year: 2085-2086
Processing year range: 2085-2086
Months: MAM
Processing band: 0p000mm-0p254mm
Number of files in dataset 0: 365
Loading precipitation data...


/anaconda3/envs/pyEAE/lib/python3.9/site-packages/xarray/core/accessor_dt.py:72: FutureWarning: Index.ravel returning ndarray is deprecated; in a future version this will return a view on self.
  values_as_series = pd.Series(values.ravel(), copy=False)


Preparing threshold values...
Using fixed thresholds: 0.0mm to 0.254mm
Creating masks for the precipitation band...
Calculating occurrences...
Calculating accumulations...
Saving occurrences to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/DIST_EOC4p5_2085-2086_MAM_0p000mm-0p254mm_0p254base_wet_period.nc
Saving accumulations to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/Accum_EOC4p5_2089-2090_MAM_0p000mm-0p254mm_0p254base_wet_period.nc
Processing time: 1490.25 seconds (24.84 minutes)
Current time: 22:16:20
--------------------------------------------------
Processing year: 2090-2091
Processing year range: 2090-2091
Months: MAM
Processing band: 0p000mm-0p254mm
Number of files in dataset 0: 365
Loading precipitation data...


/anaconda3/envs/pyEAE/lib/python3.9/site-packages/xarray/core/accessor_dt.py:72: FutureWarning: Index.ravel returning ndarray is deprecated; in a future version this will return a view on self.
  values_as_series = pd.Series(values.ravel(), copy=False)


Preparing threshold values...
Using fixed thresholds: 0.0mm to 0.254mm
Creating masks for the precipitation band...
Calculating occurrences...
Calculating accumulations...
Saving occurrences to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/DIST_EOC4p5_2090-2091_MAM_0p000mm-0p254mm_0p254base_wet_period.nc
Saving accumulations to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/Accum_EOC4p5_2090-2091_MAM_0p000mm-0p254mm_0p254base_wet_period.nc
Processing time: 1711.16 seconds (28.52 minutes)
Current time: 22:44:52
--------------------------------------------------
Processing year: 2091-2092
Processing year range: 2091-2092
Months: MAM
Processing band: 0p000mm-0p254mm
Number of files in dataset 0: 365
Loading precipitation data...


...

Preparing threshold values...
Using fixed thresholds: 0.0mm to 0.254mm
Creating masks for the precipitation band...
Calculating occurrences...
Calculating accumulations...
Saving occurrences to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/DIST_EOC4p5_2096-2097_MAM_0p000mm-0p254mm_0p254base_wet_period.nc
Saving accumulations to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/Accum_EOC4p5_2096-2097_MAM_0p000mm-0p254mm_0p254base_wet_period.nc
Processing time: 1501.86 seconds (25.03 minutes)
Current time: 01:23:23
--------------------------------------------------
Processing year: 2097-2098
Processing year range: 2097-2098
Months: MAM
Processing band: 0p000mm-0p254mm
Number of files in dataset 0: 365
Loading precipitation data...


/anaconda3/envs/pyEAE/lib/python3.9/site-packages/xarray/core/accessor_dt.py:72: FutureWarning: Index.ravel returning ndarray is deprecated; in a future version this will return a view on self.
  values_as_series = pd.Series(values.ravel(), copy=False)


Preparing threshold values...
Using fixed thresholds: 0.0mm to 0.254mm
Creating masks for the precipitation band...
Calculating occurrences...
Calculating accumulations...
Saving occurrences to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/DIST_EOC4p5_2097-2098_MAM_0p000mm-0p254mm_0p254base_wet_period.nc
Saving accumulations to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/Accum_EOC4p5_2097-2098_MAM_0p000mm-0p254mm_0p254base_wet_period.nc
Processing time: 1688.86 seconds (28.15 minutes)
Current time: 01:51:33
--------------------------------------------------
Processing year: 2098-2099
Processing year range: 2098-2099
Months: MAM
Processing band: 0p000mm-0p254mm
Number of files in dataset 0: 365
Loading precipitation data...


/anaconda3/envs/pyEAE/lib/python3.9/site-packages/xarray/core/accessor_dt.py:72: FutureWarning: Index.ravel returning ndarray is deprecated; in a future version this will return a view on self.
  values_as_series = pd.Series(values.ravel(), copy=False)


Preparing threshold values...
Using fixed thresholds: 0.0mm to 0.254mm
Creating masks for the precipitation band...
Calculating occurrences...
Calculating accumulations...
Saving occurrences to: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/EOC4p5/DIST_EOC4p5_2098-2099_MAM_0p000mm-0p254mm_0p254base_wet_period.nc


In [ ]:
# Optional - Visualize Results
def visualize_band_results(year, band_desc, var_type='both', season_months=None):
    """
    Create visualizations of the band results for visual validation.
    
    Parameters:
    -----------
    year : int
        Starting year of the water year
    band_desc : str
        Description of the percentile band (e.g., "0.000mm-0.254mm")
    var_type : str
        'occur', 'accum', or 'both'
    season_months : list
        List of months in the season, e.g., [3, 4, 5] for spring
    """
    if season_months is None:
        season_months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    
    # Create a string representation of months in the season
    months_dict = {1: "J", 2: "F", 3: "M", 4: "A", 5: "M", 6: "J", 
                   7: "J", 8: "A", 9: "S", 10: "O", 11: "N", 12: "D"}
    season_str = "".join([months_dict[v] for v in season_months]) if len(season_months) < 12 else "ALL"
    
    output_dir = f"/home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/{scenario}"
    
    if var_type in ['occur', 'both']:
        # Visualize occurrences
        occur_filename = f"DIST_{scenario}_{year}-{year+1}_{season_str}_{band_desc}_0p254base_wet_period.nc"
        occur_path = os.path.join(output_dir, occur_filename)
        
        if os.path.exists(occur_path):
            ds = xr.open_dataset(occur_path)
            data = ds['PRECIP_PERIOD']
            
            fig = plt.figure(figsize=(12, 8))
            ax = plt.axes(projection=ccrs.PlateCarree())
            
            # Add map features
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.5)
            ax.add_feature(cfeature.STATES, linewidth=0.3, alpha=0.5)
            
            # Plot the data
            vmax = np.ceil(data.max().values / 10) * 10  # Round up to nearest 10
            levels = np.linspace(0, vmax, 11)
            cs = plt.contourf(ds.longitude, ds.latitude, data, levels=levels, cmap='viridis', 
                             transform=ccrs.PlateCarree())
            
            # Add a colorbar
            cbar = plt.colorbar(cs, shrink=0.6, pad=0.05)
            cbar.set_label('Number of days', fontsize=12)
            
            # Add title
            plt.title(f"{scenario} Precipitation Occurrences for {band_desc} band\n{year}-{year+1}, {season_str}", 
                     fontsize=14)
            
            plt.show()
            
            ds.close()
        else:
            print(f"WARNING: Accumulations file {accum_filename} not found")
ds.close()
        else:
            print(f"WARNING: Occurrences file {occur_filename} not found")
    
    if var_type in ['accum', 'both']:
        # Visualize accumulations
        accum_filename = f"Accum_{scenario}_{year}-{year+1}_{season_str}_{band_desc}_0p254base_wet_period.nc"
        accum_path = os.path.join(output_dir, accum_filename)
        
        if os.path.exists(accum_path):
            ds = xr.open_dataset(accum_path)
            data = ds['PRECIP_SUM']
            
            fig = plt.figure(figsize=(12, 8))
            ax = plt.axes(projection=ccrs.PlateCarree())
            
            # Add map features
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.5)
            ax.add_feature(cfeature.STATES, linewidth=0.3, alpha=0.5)
            
            # Plot the data
            vmax = np.ceil(data.max().values / 100) * 100  # Round up to nearest 100
            levels = np.linspace(0, vmax, 11)
            cs = plt.contourf(ds.longitude, ds.latitude, data, levels=levels, cmap='viridis', 
                             transform=ccrs.PlateCarree())
            
            # Add a colorbar
            cbar = plt.colorbar(cs, shrink=0.6, pad=0.05)
            cbar.set_label('Precipitation (mm)', fontsize=12)
            
            # Add title
            plt.title(f"{scenario} Precipitation Accumulations for {band_desc} band\n{year}-{year+1}, {season_str}", 
                     fontsize=14)
            
            plt.show()